# 📗 질의 변환과 근거 검색

검색기는 그대로 두고 **검색에 넣는 질문**을 바꿉니다. Advanced RAG 단계표의 검색 전(Pre-retrieval) 단계입니다.

## 오늘의 목표

- [ ] Self-Query로 검색어와 조건을 나누고 BM25·Dense에 같은 조건을 적용할 수 있습니다.
- [ ] HyDE·Multi-Query·Step-back·Decomposition의 차이를 알고 질문에 맞게 고를 수 있습니다.
- [ ] 실제 검색한 원문만 근거로 원질문에 답할 수 있습니다.

## ⏪ 지난 시간 복습

교안 01에서는 BM25(어휘 일치)와 Dense(의미 유사도)의 순위를 RRF로 합쳤습니다. 시연은 같은 가상 도서 목록, 따라하기는 같은 매뉴얼 44절입니다. 모델이 만든 질문·문단은 실행마다 달라집니다.


## 다섯 가지 질의 변환

| 방법 | 바꾸는 것 | 짧은 예시 |
|---|---|---|
| **Self-Query** | 검색어와 메타데이터 조건을 분리 | ‘2020년 이후 컴퓨터 분야 딥러닝 책’ → 검색어 `딥러닝` + 연도·분류 조건 |
| **HyDE** | 가상의 답 문단을 만들고 그 임베딩으로 검색 | 장비 지원에 관한 가상 문단 → 관련된 실제 원문 검색 |
| **Multi-Query** | 같은 뜻의 질문을 여러 표현으로 작성 | ‘재택근무 장비 지원’ → ‘집에서 일할 때 장비를 지원받나요?’ |
| **Step-back** | 더 일반적인 질문을 만들어 배경 지식도 별도로 검색 | ‘우리 팀 장비 지원 기준’ → ‘재택근무 환경에서 고려할 점’ |
| **Decomposition** | 여러 정보 요구를 각각의 하위 질문으로 분리 | ‘신청과 장비 지원’ → ‘신청 절차’ + ‘장비 지원 기준’ |

HyDE는 Hypothetical Document Embeddings의 약자입니다. **생성한 질문과 가상문서는 검색 입력이며, 최종 답변의 근거는 실제 검색한 원문입니다.**

<img src="images/03_query_transformations.png" width="1100" alt="질의 변환 다섯 방법의 목적과 입력·변환·검색 예시">


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하고 준비 셀을 위에서부터 실행하세요. 경로(`material_dir`·`data_dir`·`output_dir`)와 `read_json`·`save_json`은 앞 단원과 같습니다. 첫 셀에서 `langchain-community` 유지보수 종료를 알리는 경고가 한 번 보일 수 있습니다. `BM25Retriever`를 이 패키지에서 가져오기 때문이며 실행에는 문제가 없습니다.


In [ ]:
# 문서·검색기·모델에 필요한 라이브러리를 가져옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


임베딩은 `text-embedding-3-large`의 768차원입니다. `check_embedding_ctx_length=False`는 자동 길이 검사·분할을 끕니다. API 키는 `.env`에서 읽습니다.


In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 이 셀은 모델 설정만 준비합니다. GPT 요청은 뒤의 체인 invoke에서 발생합니다.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 적재·검색 셀에서 요청합니다.")


절 하나가 이미 짧은 발췌라서 다시 나누지 않고 레코드 하나를 검색 단위 하나로 씁니다. 이 단위를 고정한 채 검색 방식과 질문을 바꿔 결과를 비교합니다. `make_documents`는 원문 ID·제목·출처에 필터용 메타데이터를 더해 `Document`를 만듭니다.


In [ ]:
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 사용하고 출처·필터 메타데이터를 담습니다.
    return [Document(
        id=record["doc_id"],
        page_content=record["text"],
        metadata={"source_id": record["doc_id"], "title": record["title"],
                  "url": record["url"], "source": record["source"], **record["metadata"]},
    ) for record in records]


In [ ]:
# 전체 목록을 먼저 보고 질문에 필요한 필터 필드를 확인합니다.
records = read_json("demo_docs.json")
display(pd.DataFrame(records)[["doc_id", "title", "metadata"]])
print(records[0]["text"])
documents = make_documents(records)


In [ ]:
def show_results(documents):
    """앞 5개 결과의 원문 ID·메타데이터·본문을 모든 열과 함께 보여 줍니다."""
    # 빈 결과는 필터를 바꾸지 않고 그대로 알립니다.
    if not documents:
        print("조건에 맞는 검색 결과가 없습니다.")
        return
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    # 열 순서를 고정해야 표를 나란히 비교할 수 있습니다. url은 길어서 뺍니다.
    rows = [{"display_order": order, "source_id": doc.metadata["source_id"], "title": doc.metadata["title"],
             **{key: doc.metadata[key] for key in sorted(doc.metadata) if key not in {"source_id", "title", "url"}},
             "text": doc.page_content} for order, doc in enumerate(documents, start=1)]
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head())


In [ ]:
# 질의 생성과 메타데이터 조건 번역에 쓰는 도구를 가져옵니다.
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from pydantic import BaseModel, Field


In [ ]:
# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF 표기 통일: 재택･원격근무 → 재택·원격근무
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    return [token.form.lower() for token in kiwi.tokenize(text)
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


In [ ]:
# 이 교안도 원문·필터 메타데이터부터 읽고 시작합니다.
practice_records = read_json("practice_docs.json")
practice_documents = make_documents(practice_records)
show_results(practice_documents)


In [ ]:
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day47_lesson02_books", embedding_function=embedding_model)
vector_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = vector_store.add_documents(documents)
print("day47_lesson02_books 적재 수:", len(added_ids))


In [ ]:
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
practice_store = Chroma(collection_name="day47_lesson02_hr", embedding_function=embedding_model)
practice_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = practice_store.add_documents(practice_documents)
print("day47_lesson02_hr 적재 수:", len(added_ids))


In [ ]:
# 각 검색기의 후보 수와 질문별 최종 선택 개수를 한곳에서 정합니다.
TOP_K = 3

# 도서 검색기: BM25·Dense가 각각 최대 TOP_K개를 반환하고 RRF로 합칩니다.
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=TOP_K)
dense = vector_store.as_retriever(search_kwargs={"k": TOP_K})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")

# 매뉴얼 검색기: 같은 설정으로 별도의 PDF 문서 집합을 검색합니다.
practice_bm25 = BM25Retriever.from_documents(practice_documents, preprocess_func=kiwi_tokenize, k=TOP_K)
practice_dense = practice_store.as_retriever(search_kwargs={"k": TOP_K})
practice_hybrid = EnsembleRetriever(
    retrievers=[practice_bm25, practice_dense], weights=[0.5, 0.5], c=60, id_key="source_id",
)


## 1. Self-Query로 검색어와 조건을 나눕니다

**Self-Query Retrieval**(검색어·조건 추출)은 질문을 **본문 검색어와 메타데이터 조건**으로 나눕니다. ‘2018년 이전 과학 분야에서 천체의 일생과 생물의 변화를 설명하는 책’은 주제 검색어와 `category = 과학`, `year < 2018` 조건으로 나눕니다.

**유용한 경우:** 분류·연도 같은 조건과 본문에서 찾을 주제가 한 질문에 함께 들어 있을 때.

`AttributeInfo`로 필드의 이름·자료형·설명을, `schema_prompt`로 추출 규칙을 알려 줍니다. `description`에 적은 분류 목록은 LLM에 주는 안내이며 자동 검증 규칙은 아닙니다.

**질문 → 검색어·조건 추출 → Chroma에서 조건에 맞는 문서 검색**

`SelfQueryRetriever`는 벡터 저장소 전용이라 `retriever=hybrid`를 받지 않습니다. `query_constructor.invoke`로 검색어·조건을 한 번 추출하고 `ChromaTranslator`로 필터를 만듭니다. `similarity_search(query_text, filter=where)`는 그 조건을 만족하는 문서 중 검색어와 가까운 문서를 찾습니다. 다음 절에서 BM25에도 같은 조건을 적용해 하이브리드 검색으로 답합니다.


<img src="images/06_self_query_flow.png" width="1100" alt="검색어는 본문 비교에, 조건은 검색 대상 제한에 사용합니다. 두 검색의 결과를 합쳐 원질문에 답합니다.">

검색어는 본문 비교에, 조건은 검색 대상 제한에 사용합니다. 두 검색의 결과를 합쳐 원질문에 답합니다.


In [ ]:
# 검색어와 조건을 추출하는 규칙입니다.
query_schema_prompt = PromptTemplate.from_template(
    "마지막 User Query를 검색어와 메타데이터 조건으로 나누세요. "
    "query에는 본문에서 찾을 핵심 주제를, filter에는 명시된 조건을 넣습니다. "
    "조건이 없을 때만 filter를 NO_FILTER로 적으세요. "
    "filter는 비교 연산자({allowed_comparators})와 논리 연산자({allowed_operators})로 표현합니다. "
    "Data Source에 정의된 필드만 사용하며, 숫자는 문자열로 바꾸지 마세요."
)

# 검색 조건으로 사용할 메타데이터 필드를 알려 줍니다.
metadata_field_info = [
    AttributeInfo(name="category", description="장서 분류. 컴퓨터, 수학, 과학, 역사, 문학 중 하나", type="string"),
    AttributeInfo(name="year", description="가상 장서의 출판 연도", type="integer"),
]

self_query = SelfQueryRetriever.from_llm(
    llm=llm, vectorstore=vector_store, document_contents="수업용 가상 도서 목록의 제목과 소개문",
    metadata_field_info=metadata_field_info,
    chain_kwargs={"schema_prompt": query_schema_prompt},
    structured_query_translator=ChromaTranslator(), search_kwargs={"k": TOP_K},
)


In [ ]:
# 우주와 별(book05)·생물의 진화(book06)를 다룬 두 과학책을 정답으로 정합니다.
question = "2018년 이전 과학 분야에서 천체의 일생과 생물이 오랜 시간에 걸쳐 달라지는 과정을 설명하는 책들을 찾아 주세요."
expected_ids = {"book05", "book06"}
parsed = self_query.query_constructor.invoke({"query": question})
print("추출 결과:", parsed)

query_text, search_kwargs = ChromaTranslator().visit_structured_query(parsed)
where = search_kwargs.get("filter")
print("검색어:", query_text)
print("조건:", where)


In [ ]:
# 추출한 검색어와 조건을 재사용해 실제 Chroma 컬렉션을 검색합니다.
self_query_results = vector_store.similarity_search(query_text, k=TOP_K, filter=where)
show_results(self_query_results)


### 🖐️ 함께 따라하기: PDF의 페이지 조건 추출하기

필드는 `category`(공통·시차출퇴근·선택근무·재택근무)와 `pdf_page`(정수)입니다. 추출 규칙을 담은 **query_schema_prompt**와 **practice_metadata**, **practice_self_query**를 만드세요. `chain_kwargs`에는 `schema_prompt`를 전달합니다.

‘공통 분류이고 PDF 20쪽 이하인 문서에서 신청 자격, 승인 판단, 승인 후 유연근무 해제 절차를 찾아 주세요.’를 추출해 **practice_query_text**, **practice_where**에 담으세요. 추출한 값으로 `practice_store.similarity_search`를 호출해 **practice_self_query_results**에 담고 출력하세요. 검색된 문서가 공통 분류의 20쪽 이하인지 확인합니다. **practice_expected_ids**는 `{hr_eligibility, hr_approval, hr_management}`로 정합니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 2. 같은 조건 안에서 하이브리드로 검색합니다

**Hybrid Search**(하이브리드 검색)에 같은 조건을 적용합니다. BM25 검색기는 한 번만 만듭니다. **전체 문서 점수 계산 → 조건에 맞는 문서만 남김 → 상위 K개 선택** 순서로 검색합니다. IDF와 평균 문서 길이는 처음 구축한 전체 문서 기준을 유지합니다.

**유용한 경우:** 특정 분류·기간으로 검색 대상을 제한하면서, 정확한 용어 일치와 의미 검색을 함께 활용할 때.

`Chroma.get(where=...)`로 조건에 맞는 ID를 얻고, Dense에도 같은 조건을 `filter`로 적용합니다. 두 결과는 기존 `hybrid`의 **RRF**(Reciprocal Rank Fusion, 역순위 융합)로 합칩니다. **상위 K개를 먼저 뽑고 필터링하면** 그 아래의 조건 일치 문서를 놓칠 수 있습니다.


<img src="images/02_shared_filter.png" width="1100" alt="조건을 한 번 추출해 BM25와 Dense에 함께 적용합니다.">

조건을 한 번 추출해 BM25와 Dense에 함께 적용합니다.


In [ ]:
def search_with_filter(query, where, bm25, vector_store, hybrid, k):
    """기존 BM25를 재사용하고 같은 조건의 Dense 결과와 합칩니다."""
    # 두 검색에 적용할 조건 일치 문서 ID를 Chroma에서 조회합니다.
    matched = vector_store.get(where=where, include=["metadatas"])
    allowed_ids = {item["source_id"] for item in matched["metadatas"]}
    if not allowed_ids:
        return []
    if not query.strip():
        candidates = [doc for doc in bm25.docs if doc.metadata["source_id"] in allowed_ids]
        return sorted(candidates, key=lambda doc: doc.metadata["source_id"])[:k]

    # BM25 점수를 계산합니다.
    scores = bm25.vectorizer.get_scores(bm25.preprocess_func(query))

    # 조건에 맞는 문서와 점수만 남깁니다.
    scored_candidates = [
        (doc, score) for doc, score in zip(bm25.docs, scores)
        if doc.metadata["source_id"] in allowed_ids
    ]

    # 조건에 맞는 문서를 BM25 점수 내림차순으로 정렬해 상위 k개를 선택합니다.
    scored_candidates.sort(key=lambda item: item[1], reverse=True)
    bm25_results = [doc for doc, score in scored_candidates[:k]]

    # Dense도 같은 조건을 적용해 실제 벡터 DB에서 검색합니다.
    dense_results = vector_store.similarity_search(query, k=k, filter=where)

    # 문서별 가중 RRF 점수를 합산해 내림차순으로 정렬하고 상위 k개를 반환합니다.
    return hybrid.weighted_reciprocal_rank([bm25_results, dense_results])[:k]


### Recall로 변환 전후 비교하기

**Recall = 찾은 정답 문서 수 ÷ 전체 정답 문서 수**입니다. 원문을 읽고 질문마다 정답 ID 2~3개를 미리 정합니다. 두 개 중 하나만 찾으면 0.5, 세 개 중 두 개를 찾으면 약 0.667입니다.

`retrieved_count`는 중복을 뺀 검색 문서 수, `expected_count`는 정답 문서 수, `missing_ids`는 놓친 정답입니다. Recall은 답변 정확도가 아니므로 LLM 답변의 인용 원문도 읽어 확인합니다.

Self-Query·HyDE는 최종 상위 TOP_K개를, 나머지는 여러 검색의 결과를 합친 전체 후보를 평가합니다. **후보 수가 다르므로 방법 간 동일 K 성능 비교는 아닙니다.** Multi-Query·Step-back은 원질문 결과를 포함해 Recall이 줄지 않는 구조입니다. `show_results`는 앞 5개만 보여 주지만 평가는 전체 목록을 사용합니다.

이 예시들은 변환으로 놓친 문서를 찾는 과정을 관찰하기 위한 것입니다. 실제 성능은 별도의 질문 모음으로 평가합니다.


In [ ]:
def compare_recall(before, after, expected_ids):
    """변환 전후의 전체 후보 Recall과 중복을 제외한 문서 수를 비교합니다."""
    expected_ids = set(expected_ids)
    rows = []
    for stage, results in [("변환 전", before), ("변환 후", after)]:
        retrieved_ids = {doc.metadata["source_id"] for doc in results}
        rows.append({
            "stage": stage,
            "recall": len(retrieved_ids & expected_ids) / len(expected_ids),
            "retrieved_count": len(retrieved_ids),
            "expected_count": len(expected_ids),
            "missing_ids": sorted(expected_ids - retrieved_ids),
        })
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head().round(3))


### 실제 검색 문서로 답변하기

모든 방법에서 **BM25 + Chroma의 벡터 검색 → RRF 결합 → 원질문에 답변**까지 실행합니다. 가중치는 비교용으로 `0.5/0.5`로 고정합니다. 생성한 질문·가상문서는 검색에만 쓰고, 답변에는 검색된 원문과 출처 ID를 전달합니다.


In [ ]:
# 본문과 조건 메타데이터를 함께 줘 연도·분류·쪽수도 답변에서 확인할 수 있게 합니다.
def format_context(documents):
    """실제 검색 문서의 본문·메타데이터·출처를 답변 문맥으로 만듭니다."""
    return "\n\n".join(
        f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n"
        f"메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for doc in documents
    )


In [ ]:
# 실제 검색 원문에 있는 내용으로만 원질문에 답합니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 본문과 메타데이터에서 직접 확인되는 내용만 한글로 답하세요. "
               "질문에 직접 답하는 내용만 최대 3개 항목으로 쓰고, 항목마다 1~2문장과 [원문 ID] 인용을 넣으세요. 관련 없는 문서는 언급하지 마세요. "
               "원문에 없는 절차·조건·조언을 추가하지 마세요. "
               "원문의 권고나 가능성을 의무로 바꾸지 말고, 수치·단위·비율·조건을 그대로 보존하세요. 사용자가 제시한 방안을 원문이 허용하거나 정당화한다고 추론하지 마세요. "
               "근거가 부족한 부분은 확인할 수 없다고 말하고, 추가 결론은 쓰지 마세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()


In [ ]:
# 원질문과 추출한 검색어·조건을 확인합니다.
print("원질문:", question)
print("검색어:", query_text)
print("조건:", where)

# 한 번 추출한 검색어와 조건을 그대로 사용합니다.
filtered_results = search_with_filter(query_text, where, bm25, vector_store, hybrid, k=TOP_K)
show_results(filtered_results)

# 원질문 검색과 검색어·조건 추출 후 검색의 Recall을 비교합니다.
self_query_before = hybrid.invoke(question)[:TOP_K]
compare_recall(self_query_before, filtered_results, expected_ids)


In [ ]:
# 검색된 실제 문서로 원질문에 답합니다.
self_query_answer = answer_chain.invoke({
    "question": question, "context": format_context(filtered_results),
})
print(self_query_answer)


### 🖐️ 함께 따라하기: 같은 PDF 조건을 양쪽에 적용하기

1번 따라하기의 `practice_query_text`·`practice_where`와 준비한 `practice_bm25`·`practice_store`·`practice_hybrid`를 `search_with_filter`에 전달해 **practice_filtered**에 담으세요. 이 원문으로 원질문에 답한 **practice_self_query_answer**를 출력하세요.

원질문의 상위 TOP_K개를 **practice_self_query_before**에 담고 `compare_recall`로 전후를 비교하세요. 정답은 신청 자격(`hr_eligibility`, 18쪽), 승인 판단(`hr_approval`, 18쪽), 유연근무 해제(`hr_management`, 19쪽)를 다룬 3개 원문입니다.

**확인 기준**: 공통 분류의 20쪽 이하에서 검색하고, 세 요구에 필요한 원문을 찾았는지 확인합니다. 조건 적용 전에는 재택근무·시차출퇴근의 신청 절차도 검색될 수 있습니다. 답변은 검색된 원문을 인용합니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 3. HyDE의 가상문서로 실제 문서를 찾습니다

**HyDE**(Hypothetical Document Embeddings, 가상문서 임베딩)는 가상의 답 문단을 임베딩해 실제 원문을 찾는 방법입니다. [원래 HyDE](https://arxiv.org/abs/2212.10496)는 Dense 검색 기법입니다.

**유용한 경우:** 짧은 질문만으로 설명형 문서를 찾기 어려워, 답변에 쓰일 핵심 개념을 검색에 보충해야 할 때.

여기서는 **원질문 → BM25**, **가상문서 → Chroma 벡터 검색**의 두 결과를 RRF로 합칩니다. 가상문서에 잘못된 내용이 생길 수 있으므로, 답변은 실제 검색된 원문만 근거로 작성합니다.


<img src="images/08_hyde_flow.png" width="1100" alt="HyDE를 하이브리드에 결합한 흐름입니다. BM25에는 원질문, Dense에는 가상문서를 넣습니다.">

HyDE를 하이브리드에 결합한 흐름입니다. BM25에는 원질문, Dense에는 가상문서를 넣습니다.


In [ ]:
# 가상문서는 검색에만 쓰고 최종 답변은 실제 원문으로 작성합니다.
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "질문에 답할 내용이 담긴 가상의 원문을 3문장 이내로 작성하세요. "
               "각 요구에 필요한 핵심 개념을 해당 분야의 표준 용어로 명시하고 설명하세요. "
               "질문을 다시 쓰거나 관련 주제를 추가하지 마세요. "
               "도서 검색 질문에는 해당 내용을 가르치는 책의 소개문을 쓰되 책 제목·저자·출판사는 만들지 마세요. "
               "확인되지 않은 구체적인 수치나 규정을 지어내지 마세요. "
               "요청문·검색어 목록 없이 한글 원문만 출력하세요."),
    ("human", "{question}"),
])
hyde_chain = hyde_prompt | llm | StrOutputParser()


In [ ]:
# 표 결합·집계(book02)와 설문 해석(book03)을 익힐 책을 찾습니다.
question = "직원 설문 보고서를 맡았습니다. 응답자 정보와 답변 표를 연결해 부서별 수치를 만들고, 일부 직원의 답변을 회사 전체의 경향으로 해석할 때 주의할 점을 배울 책을 찾아 주세요."
expected_ids = {"book02", "book03"}

# 질문에 답할 내용을 담은 가상의 책 소개문을 생성합니다.
hypothetical_document = hyde_chain.invoke({"question": question})
print("검색용 가상문서:", hypothetical_document)


In [ ]:
# BM25는 원질문으로, Dense는 가상문서로 검색합니다.
hyde_bm25_results = bm25.invoke(question)
hyde_dense_results = dense.invoke(hypothetical_document)

# 문서별 가중 RRF 점수를 합산해 내림차순으로 정렬하고 상위 TOP_K개를 선택합니다.
hyde_results = hybrid.weighted_reciprocal_rank(
    [hyde_bm25_results, hyde_dense_results]
)[:TOP_K]
show_results(hyde_results)


In [ ]:
# 두 방식 모두 상위 TOP_K개 문서의 Recall을 계산합니다.
hyde_before = hybrid.invoke(question)[:TOP_K]
show_results(hyde_before)
compare_recall(hyde_before, hyde_results, expected_ids)


In [ ]:
# 검색된 실제 문서로 원질문에 답합니다.
hyde_answer = answer_chain.invoke({
    "question": question, "context": format_context(hyde_results),
})
print(hyde_answer)


### 🖐️ 함께 따라하기: PDF와 가상문서 구분하기

‘집에서 일하는 팀에 성과가 낮은 직원은 바로 사무실로 복귀시키고 잡담은 금지하려고 합니다. 이렇게 운영해도 괜찮나요?’로 `practice_hypothesis`를 생성하세요. 원질문의 BM25 결과와 가상문서의 Dense 결과를 RRF로 합쳐 상위 TOP_K개를 `practice_hyde_results`에 담고, 실제 원문으로 `practice_hyde_answer`를 생성하세요.

정답 `practice_expected_ids`는 성과 부진 대응 `hr_feedback`, 팀의 대화 방식 `hr_collaboration_team_trust`입니다. 원질문의 상위 TOP_K개 `practice_hyde_before`와 `compare_recall`로 비교하세요.

**확인 기준**: 가상문서로 새로 찾은 정답을 확인하고, 최종 답변이 실제 원문의 권고를 정확히 설명하는지 읽습니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 4. Multi-Query로 같은 뜻의 표현을 늘립니다

**Multi-Query Retrieval**(다중 질의 검색)은 같은 뜻의 질문을 여러 표현으로 만들어 검색 범위를 넓힙니다. `MultiQueryRetriever`에 `hybrid`를 넣으면 **생성 질문마다 BM25·Chroma를 검색**합니다. `include_original=True`로 원질문도 검색합니다.

**유용한 경우:** ‘재택근무’와 ‘집에서 일하기’처럼 같은 뜻의 표현이 다양해, 한 가지 표현으로 관련 문서를 놓칠 때.

`MultiQueryRetriever`에는 별도 `top_k` 인자가 없습니다. 내부 BM25·Dense의 `k=TOP_K`는 **질문마다 각 검색기가 가져올 개수**입니다. 두 결과와 여러 질문의 결과를 합치므로 최종 문서는 `TOP_K`개를 넘을 수 있습니다.

`multi_query | unique_documents`로 검색과 원문 ID 기준 중복 제거를 연결합니다. `MultiQueryRetriever` 자체도 같은 `Document`를 제거하지만, 여기서는 원문 ID를 기준으로 통일합니다. 합친 목록 전체의 순서는 관련성 순위가 아닙니다.


<img src="images/07_multi_query_flow.png" width="1100" alt="같은 뜻의 여러 질문과 원질문을 각각 하이브리드 검색하고, 중복 원문을 제거해 답합니다.">

같은 뜻의 여러 질문과 원질문을 각각 하이브리드 검색하고, 중복 원문을 제거해 답합니다.


In [ ]:
# 여러 질문이 같은 원문을 찾아도 답변 문맥에는 한 번만 넣습니다.
def unique_documents(documents):
    """원문 ID를 기준으로 첫 등장 순서를 유지하며 중복을 제거합니다."""
    by_id = {}
    for doc in documents:
        source_id = doc.metadata["source_id"]
        if source_id not in by_id:
            by_id[source_id] = doc
    return list(by_id.values())


In [ ]:
# from_llm의 기본 줄 단위 파서가 읽을 수 있게 번호 없이 한 줄에 하나씩 받습니다.
multi_prompt = ChatPromptTemplate.from_template(
    "질문과 같은 의미를 유지하는 검색 질문을 서로 다른 표현으로 정확히 3개 작성하세요. "
    "일상 표현을 관련 분야의 표준 용어로 바꾼 질문도 포함하세요. "
    "원문의 정보 요구·고유명사·조건을 유지하세요. 질문의 언어를 유지하세요. "
    "설명·번호·빈 줄 없이 한 줄에 질문 하나만 출력하세요.\n질문: {question}"
)


In [ ]:
def show_queries(queries):
    """실제로 검색에 사용할 생성 질문을 출력하고 그대로 전달합니다."""
    # 출력용으로 LLM을 다시 호출하지 않고 기존 체인의 결과를 관찰합니다.
    print("생성 질문:", queries)
    return queries


In [ ]:
# 원질문도 함께 검색해 생성 질문이 놓친 표현을 보완합니다.
multi_query = MultiQueryRetriever.from_llm(
    retriever=hybrid, llm=llm, prompt=multi_prompt, include_original=True,
)

# 생성 질문을 출력합니다. 원질문은 검색기가 추가합니다.
multi_query.llm_chain = multi_query.llm_chain | show_queries

# 질문별 하이브리드 검색 뒤 원문 ID로 중복을 제거합니다.
multi_query_chain = multi_query | unique_documents


In [ ]:
# 변환하지 않은 원질문의 하이브리드 결과를 먼저 확인합니다.
# 천체의 일생(book05)·생물의 진화(book06)를 다룬 책을 찾습니다.
question = "항성이 태어나 사라지는 과정과 생물 종이 오랜 세월에 걸쳐 달라지는 과정을 배울 책들을 찾아 주세요."
expected_ids = {"book05", "book06"}
multi_before = hybrid.invoke(question)
show_results(multi_before)


In [ ]:
# 생성 질문과 원질문의 결과를 같은 문서 없이 합친 목록입니다.
multi_results = multi_query_chain.invoke(question)
show_results(multi_results)

compare_recall(multi_before, multi_results, expected_ids)


In [ ]:
# 검색된 실제 문서로 원질문에 답합니다.
multi_answer = answer_chain.invoke({
    "question": question, "context": format_context(multi_results),
})
print(multi_answer)


### 🖐️ 함께 따라하기: PDF의 다른 표현 함께 찾기

‘직원마다 집에서 일할 수 있는지 판단할 기준을 마련하려 합니다. 업무 점검표에서 살펴볼 항목, 혼자 일할 경력을 신청 자격에 반영할 방법, 허락 여부를 판단할 담당자와 검토 사항을 알려 주세요.’를 `practice_hybrid`로 검색하세요. `practice_multi`에 원질문을 포함하고 `show_queries`를 연결한 뒤, 중복 ID를 제거한 **practice_multi_results**로 **practice_multi_answer**를 생성하세요.

정답 **practice_expected_ids**는 업무 점검표 `hr_job_review`, 신청 자격 `hr_eligibility`, 승인 담당자·판단 요소 `hr_approval`입니다. **practice_multi_before**와 `compare_recall`로 비교하세요.

**확인 기준**: 생성 질문마다 세 요구가 유지되고, 원질문에서 놓친 정답을 추가로 찾았는지 확인합니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 5. Step-back으로 배경 지식도 검색하기

**Step-back Prompting**(배경 질문 생성)은 구체적인 질문에서 관련 원리·개념을 묻는 배경 질문을 만듭니다.

**유용한 경우:** 구체적인 문제의 답뿐 아니라 그 이유를 설명할 원리·정의까지 함께 찾아야 할 때.

**원질문과 배경 질문을 각각 하이브리드 검색 → 원문을 합쳐 원질문에 답하기**

원질문은 구체적인 근거를, 배경 질문은 더 넓은 설명을 찾습니다. [원논문의 RAG 실험](https://arxiv.org/html/2310.06117v2#A4.SS2)도 두 질문을 별도로 검색합니다. `hybrid.batch([원질문, 배경질문])`으로 각각 검색한 뒤 상위 TOP_K개씩 모아 중복을 제거합니다.

Multi-Query는 같은 뜻의 표현을 늘리고, Step-back은 답을 이해하는 데 필요한 배경 원리를 묻습니다. 예를 들어 가중치 갱신 질문에서 도함수·기울기의 원리로 범위를 넓힙니다.


<img src="images/09_step_back_flow.png" width="1100" alt="구체적인 원질문과 일반적인 배경 질문을 각각 검색해 답변에 함께 사용합니다.">

구체적인 원질문과 일반적인 배경 질문을 각각 검색해 답변에 함께 사용합니다.


In [ ]:
# 구체적 사례를 배경 원리로 바꾸는 예시를 함께 보여 줍니다.
stepback_prompt = ChatPromptTemplate.from_messages([
    ("system", "원질문의 구체적인 대상과 작업을 제거하고, 그 배경이 되는 기초 개념 자체를 묻는 질문 하나를 작성하세요. "
               "핵심 개념의 이름을 명시하세요. "
               "원래 대상의 처리 방법을 다른 말로 다시 묻지 마세요. "
               "도서 추천 요청이나 답변 없이 한글 질문만 출력하세요."),
    ("human", "자동차의 이동 거리를 시간별로 기록했을 때 특정 순간의 속도는 어떻게 구하나요?"),
    ("ai", "도함수와 순간 변화율은 어떤 관계이며, 함수의 변화를 어떻게 나타내나요?"),
    ("human", "{question}"),
])
stepback_chain = stepback_prompt | llm | StrOutputParser()


In [ ]:
# 원질문의 배경 개념을 묻는 질문을 생성합니다.
# 신경망 학습(book01)과 그 계산에 쓰이는 변화율(book04)을 함께 공부합니다.
question = "예측 모델의 학습 코드를 처음 맡았습니다. 오차를 줄이도록 내부 값을 바꾸는 과정을 기초부터 구현하고, 여기에 미분이 왜 쓰이는지 이해할 책을 찾아 주세요."
expected_ids = {"book01", "book04"}
background_question = stepback_chain.invoke({"question": question})
print("배경 질문:", background_question)


In [ ]:
# 각 검색기가 TOP_K개씩 찾은 뒤, RRF로 합친 최종 TOP_K개만 선택합니다.
direct_results, background_results = [
    results[:TOP_K] for results in hybrid.batch([question, background_question])
]

# 배경 질문이 새로 찾은 문서만 확인합니다.
direct_ids = {doc.metadata["source_id"] for doc in direct_results}
added_ids = {doc.metadata["source_id"] for doc in background_results} - direct_ids
print("배경 질문으로 추가된 원문 ID:", added_ids)


In [ ]:
# 원질문 결과를 앞에, 배경 결과를 뒤에 둔 합집합입니다.
stepback_results = unique_documents(direct_results + background_results)
show_results(stepback_results)

compare_recall(direct_results, stepback_results, expected_ids)


In [ ]:
# 검색된 실제 문서로 원질문에 답합니다.
stepback_answer = answer_chain.invoke({
    "question": question, "context": format_context(stepback_results),
})
print(stepback_answer)


### 🖐️ 함께 따라하기: 매뉴얼의 구체 질문과 배경 연결하기

‘시차출퇴근을 처음 도입할 때 직원별 출근 시각을 미리 고정하자는 의견입니다. 이 방식의 장점과, 이후 선택권을 넓혀도 될지 판단할 성과 측정 방법을 알려 주세요.’로 시연과 같은 순서를 밟아 `practice_background`, `practice_direct_results`, `practice_background_results`, `practice_stepback_results`, **practice_added_ids**를 만드세요. 두 질문을 각각 `practice_hybrid`로 검색해 상위 TOP_K개씩 합치고, 원질문에 답한 **practice_stepback_answer**를 출력하세요.

정답 **practice_expected_ids**는 고정형의 장점 `hr_stagger_models`, 효과 평가 방법 `hr_effect`입니다. `compare_recall`로 원질문 결과와 합친 결과를 비교하세요.

**확인 기준**: 원질문 결과는 유지되며, 배경 검색으로 정답 문서와 전체 문서가 각각 몇 개 늘었는지 확인합니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 6. 복합 질문을 나누고 실제 근거로 답합니다

**Query Decomposition**(질문 분해)은 여러 정보 요구를 독립적인 하위 질문으로 나눠 각각 하이브리드 검색합니다. Multi-Query가 같은 뜻의 표현을 늘린다면, 이 방법은 서로 다른 요구를 나눕니다.

**유용한 경우:** ‘신청 절차와 장비 비용’처럼 서로 다른 요구가 섞여, 각 항목의 근거를 따로 찾아야 할 때.

`hybrid.batch(하위질문목록)`으로 각 질문을 검색합니다. 검색된 원문의 중복을 제거한 뒤, 원질문 전체에 답합니다. 원문에 없는 세부 내용은 확인할 수 없다고 답해야 합니다.


<img src="images/10_decomposition_flow.png" width="1100" alt="서로 다른 정보 요구를 나눠 검색하고, 각 근거를 모아 원질문 전체에 답합니다.">

서로 다른 정보 요구를 나눠 검색하고, 각 근거를 모아 원질문 전체에 답합니다.


In [ ]:
class SubQuestions(BaseModel):
    """원질문을 나누어 검색할 하위 질문 목록입니다."""
    questions: list[str] = Field(
        description=(
            "원질문의 서로 다른 정보 요구를 하나씩 묻는 질문 2~3개. "
            "각 질문만으로 검색할 수 있게 대상과 조건을 포함하고, "
            "원질문에 없는 요구를 추가하지 않으며 같은 언어로 작성."
        ),
        min_length=2, max_length=3,
    )


# 스키마는 목록 모양을 제어합니다. 질문이 원의도를 보존했는지는 사람이 읽습니다.
decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", "원질문의 서로 다른 정보 요구를 하나씩 묻는 질문 2~3개로 나누세요. "
               "각 질문만으로 검색할 수 있게 대상과 조건을 포함하세요. 원질문에 없는 요구를 추가하지 말고 같은 언어로 작성하세요."),
    ("human", "{question}"),
])
decompose_chain = decompose_prompt | llm.with_structured_output(
    SubQuestions, method="json_schema",
)


In [ ]:
# 원질문을 하위 질문 2~3개로 나눕니다.
# 표 결합(book02)·항성의 일생(book05)·생물의 협력(book06)에 각각 한 권이 필요합니다.
question = "데이터 표를 이어 붙이는 방법, 항성의 일생, 생물이 남을 돕는 이유를 각각 배울 책을 찾아 주세요."
expected_ids = {"book02", "book05", "book06"}
subquestions = decompose_chain.invoke({"question": question})
print(subquestions.questions)


In [ ]:
# 하위 질문마다 검색한 최종 TOP_K개 문서를 하나의 목록으로 모읍니다.
subquestion_results = hybrid.batch(subquestions.questions)
all_results = [
    doc
    for results in subquestion_results
    for doc in results[:TOP_K]
]


In [ ]:
# 여러 하위 질문이 찾은 같은 원문은 답변 문맥에 한 번만 넣습니다.
decomposition_results = unique_documents(all_results)
show_results(decomposition_results)

decomposition_before = hybrid.invoke(question)[:TOP_K]
compare_recall(decomposition_before, decomposition_results, expected_ids)


In [ ]:
# 답변 질문은 하위 질문이 아닌 원질문이고, 문맥은 실제 검색 결과만으로 만듭니다.
context = format_context(decomposition_results)
answer = answer_chain.invoke({"question": question, "context": context})
print(answer)


### 🖐️ 함께 따라하기: PDF의 세 가지 운영 질문 나누기

‘선택근무에서 이번 정산기간에 더 일한 시간을 다음 정산기간에 덜 일하는 것으로 맞춰도 되나요? 재택근무 중 성과가 낮으면 바로 출근하게 해도 되나요? 집에서 일할 장비와 비용은 누가 부담하나요?’로 시연과 같은 순서를 밟아 `practice_subquestions`, 합친 원문 `practice_results`, 원질문에 답한 `practice_answer`를 만드세요.

정답 **practice_expected_ids**는 `{hr_select_settlement_balance, hr_feedback, hr_environment}`입니다. 원질문의 상위 TOP_K개 **practice_decomposition_before**와 분해 후 결과를 `compare_recall`로 비교하세요.

**확인 기준**: 하위 질문이 정산시간·성과 부진·장비 비용을 빠짐없이 다루고, 답변이 실제 원문을 인용합니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 이번 강의 정리

| 기법 | 쓰는 때 | 맞지 않는 때 |
|---|---|---|
| Self-Query | 분류·연도 같은 조건이 질문에 있을 때 | 문서에 그 메타데이터가 없을 때 |
| HyDE | 질문은 짧고 문서는 설명 문장일 때 | 모델이 모르는 사내 규정·수치를 물을 때 |
| Multi-Query | 질문과 문서의 표현이 다를 때 | 정확한 용어·번호로 찾을 때 |
| Step-back | 원칙·정의를 알아야 답할 때 | 사실 하나만 확인할 때 |
| Decomposition | 정보 요구가 둘 이상일 때 | 이번 독립 분해 방식은 앞 답에 따라 다음 질문이 달라지는 경우에 부적합 |

### 방법 조합과 실무 적용

**각 방법은 함께 사용할 수 있습니다.** 예를 들어 Self-Query로 조건을 추출하고, Multi-Query로 표현을 늘린 뒤 **모든 질문에 같은 조건을 적용해 하이브리드 검색**합니다. 복합 질문은 먼저 Decomposition으로 나누어 이 과정을 적용할 수 있습니다.

**실무에서는 조건이 있는 질문에 Self-Query, 표현 차이로 검색이 잘 안 되는 질문에 Multi-Query를 우선 검토합니다.** 여러 요구가 섞인 질문에는 Decomposition을 추가합니다.

HyDE·Step-back은 위 표의 문제에 맞춰 선택적으로 추가합니다. **단계를 늘릴수록 호출 비용과 응답 시간이 늘어나므로**, 필요한 방법만 조합하고 Recall과 답변 품질을 함께 확인합니다.

## ⏭️ 다음 시간 예고

다음 시간에는 찾은 원문을 답변 전에 다시 고르고 줄이는 리랭킹·컨텍스트 압축을 다룹니다.
